# Validación del Modelo

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configuración de visualización
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

### Carga de datos

In [11]:
# Cargar el dataset agregado
df = pd.read_csv('../data_processed/datos_aggregados_dia_estacion_franja.csv')

# Convertir la columna date a datetime
df['date'] = pd.to_datetime(df['date'])

# Mostrar información general
print("Shape del dataset original:", df.shape)
print("\nPrimeras filas:")
print(df.head(10))
print("\nInformación del dataset:")
print(df.info())
print("\nValores únicos por columna:")
print(f"Franjas horarias: {df['franja_horaria'].unique()}")
print(f"Estaciones: {df['estacion'].unique()}")
print(f"Parámetros: {df['parametro'].unique()}")
print(f"Clases: {df['clase'].unique()}")

Shape del dataset original: (301694, 10)

Primeras filas:
                 date  anio  mes  dia  hora estacion franja_horaria parametro  \
0 2023-01-01 07:00:00  2023    1    1     7      SUR       pico_7_9        CO   
1 2023-01-01 08:00:00  2023    1    1     8      SUR       pico_7_9        CO   
2 2023-01-01 10:00:00  2023    1    1    10      SUR      ref_10_12        CO   
3 2023-01-01 11:00:00  2023    1    1    11      SUR      ref_10_12        CO   
4 2023-01-02 07:00:00  2023    1    2     7      SUR       pico_7_9        CO   
5 2023-01-02 08:00:00  2023    1    2     8      SUR       pico_7_9        CO   
6 2023-01-02 10:00:00  2023    1    2    10      SUR      ref_10_12        CO   
7 2023-01-02 11:00:00  2023    1    2    11      SUR      ref_10_12        CO   
8 2023-01-03 07:00:00  2023    1    3     7      SUR       pico_7_9        CO   
9 2023-01-03 08:00:00  2023    1    3     8      SUR       pico_7_9        CO   

   valor  clase  
0   1.03      0  
1   0.88      

### Filtrado de franja horaria (7-9 am)

In [12]:
# Filtrar únicamente la franja horaria pico_7_9
df_pico = df[df['franja_horaria'] == 'pico_7_9'].copy()

print(f"Registros antes del filtrado: {len(df)}")
print(f"Registros después del filtrado (7-9 am): {len(df_pico)}")
print(f"Porcentaje de datos conservados: {len(df_pico)/len(df)*100:.2f}%")
print(f"\nRango de fechas: {df_pico['date'].min()} a {df_pico['date'].max()}")
print(f"Horas incluidas: {sorted(df_pico['hora'].unique())}")

Registros antes del filtrado: 301694
Registros después del filtrado (7-9 am): 149974
Porcentaje de datos conservados: 49.71%

Rango de fechas: 2023-01-01 07:00:00 a 2025-06-30 08:00:00
Horas incluidas: [np.int64(7), np.int64(8)]


### Reestructuración del dataset en formato "wide"

In [13]:
# Reestructurar a formato wide: cada parámetro (contaminante) como columna
df_wide = df_pico.pivot_table(
    index=['date', 'anio', 'mes', 'dia', 'estacion', 'clase'],
    columns='parametro',
    values='valor',
    aggfunc='mean'  # Promediamos si hay valores duplicados
).reset_index()

# Aplanar los nombres de columnas
df_wide.columns.name = None

print("Shape del dataset en formato wide:", df_wide.shape)
print("\nPrimeras filas del dataset reestructurado:")
print(df_wide.head())
print("\nColumnas del dataset wide:")
print(df_wide.columns.tolist())
print("\nEstadísticas descriptivas de los contaminantes:")
print(df_wide.iloc[:, 6:].describe())  # Solo las columnas de contaminantes

Shape del dataset en formato wide: (10907, 11)

Primeras filas del dataset reestructurado:
                 date  anio  mes  dia   estacion  clase        CO        NO2  \
0 2023-01-01 07:00:00  2023    1    1     CENTRO      0  1.040000  13.600000   
1 2023-01-01 07:00:00  2023    1    1   NORESTE3      0  1.556333  27.100000   
2 2023-01-01 07:00:00  2023    1    1  NOROESTE3      0  1.390000  18.666667   
3 2023-01-01 07:00:00  2023    1    1     NORTE2      0  1.675000  29.400000   
4 2023-01-01 07:00:00  2023    1    1        SUR      0  1.787500  43.625000   

          O3        PM10  PM2.5  
0  16.000000   84.000000  35.00  
1   6.600000  140.666667  74.50  
2   7.333333   57.333333  21.96  
3   6.500000  162.500000  68.01  
4   6.750000   99.250000  41.44  

Columnas del dataset wide:
['date', 'anio', 'mes', 'dia', 'estacion', 'clase', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5']

Estadísticas descriptivas de los contaminantes:
                 CO           NO2            O3          PM

In [14]:
# Verificar valores faltantes por contaminante
print("\nValores faltantes por contaminante:")
contaminantes = df_wide.columns[6:]  # Columnas de contaminantes
missing_info = pd.DataFrame({
    'Contaminante': contaminantes,
    'Valores Faltantes': [df_wide[col].isna().sum() for col in contaminantes],
    'Porcentaje': [df_wide[col].isna().sum() / len(df_wide) * 100 for col in contaminantes]
})
print(missing_info)


Valores faltantes por contaminante:
  Contaminante  Valores Faltantes  Porcentaje
0           CO                 76    0.696800
1          NO2                 94    0.861832
2           O3                192    1.760337
3         PM10                 14    0.128358
4        PM2.5                606    5.556065


### Creación de la variable "temporada" (invierno / verano)

In [15]:
# Crear la variable temporada
def asignar_temporada(mes):
    """
    Asigna temporada según el mes:
    - Invierno: Enero (mes 1)
    - Verano: Agosto (mes 8)
    """
    if mes == 1:
        return 'invierno'
    elif mes == 8:
        return 'verano'
    else:
        return 'otra'

df_wide['temporada'] = df_wide['mes'].apply(asignar_temporada)

# Mostrar la distribución por temporada
print("Distribución de observaciones por temporada:")
print(df_wide['temporada'].value_counts())
print("\nDistribución porcentual:")
print(df_wide['temporada'].value_counts(normalize=True) * 100)

# Mostrar combinaciones de temporada y clase
print("\nCruce de temporada y clase (regreso a clases):")
print(pd.crosstab(df_wide['temporada'], df_wide['clase'], margins=True))

Distribución de observaciones por temporada:
temporada
otra        9049
invierno    1115
verano       743
Name: count, dtype: int64

Distribución porcentual:
temporada
otra        82.965068
invierno    10.222793
verano       6.812139
Name: proportion, dtype: float64

Cruce de temporada y clase (regreso a clases):
clase         0     1    All
temporada                   
invierno    765   350   1115
otra       5737  3312   9049
verano      486   257    743
All        6988  3919  10907


### Guardar dataset preparado con clasificación de temporada

In [17]:
# Guardar el dataset
df_wide.to_csv('../data_processed/dataset_wide_con_temporada.csv', index=False)
print("✓ Dataset guardado en: ../data_processed/dataset_wide_con_temporada.csv")
print(f"\nDimensiones del dataset guardado: {df_wide.shape}")
print(f"Columnas incluidas: {df_wide.columns.tolist()}")

✓ Dataset guardado en: ../data_processed/dataset_wide_con_temporada.csv

Dimensiones del dataset guardado: (10907, 12)
Columnas incluidas: ['date', 'anio', 'mes', 'dia', 'estacion', 'clase', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'temporada']
